# Train Personalized DeepVQE trên Google Colab

Notebook này dùng dataset đã tạo bởi `data_prep_colab.ipynb`, tự chia `metadata.json` thành train/valid manifest, rồi chạy `train_personalized.py`.

## 1. Kiểm tra GPU và mount Google Drive

In [ ]:
import os
import sys
from pathlib import Path

import torch

print(f"Python: {sys.version}")
print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

from google.colab import drive
drive.mount('/content/drive')

## 2. Cài thư viện

Colab thường đã có sẵn `torch` và `torchaudio`; notebook không ép cài `torch==1.11.0` để tránh lỗi không tương thích Python/CUDA mới.

In [ ]:
!pip install -q einops==0.7.0 speechbrain soundfile tqdm

import torchaudio
print(f"Torchaudio: {torchaudio.__version__}")

## 3. Trỏ tới source code và dataset

In [ ]:
# Nếu bạn đặt project trong Google Drive, sửa đường dẫn này cho đúng.
PROJECT_DIR = Path('/content/drive/MyDrive/deepvqe')

# Dataset mặc định khớp với data_prep_colab.ipynb.
DATASET_DIR = Path('/content/drive/MyDrive/TSE_Dataset/mixed_dataset')
METADATA_PATH = DATASET_DIR / 'metadata.json'

# Nếu muốn clone từ GitHub thay vì dùng Drive, điền URL rồi bật USE_GIT_CLONE=True.
USE_GIT_CLONE = False
GIT_REPO_URL = ''

if USE_GIT_CLONE:
    assert GIT_REPO_URL, 'Hãy điền GIT_REPO_URL trước khi clone.'
    clone_dir = Path('/content/deepvqe')
    if not clone_dir.exists():
        !git clone {GIT_REPO_URL} {clone_dir}
    PROJECT_DIR = clone_dir

assert (PROJECT_DIR / 'train_personalized.py').exists(), f'Không thấy train_personalized.py trong {PROJECT_DIR}'
assert METADATA_PATH.exists(), f'Không thấy metadata.json tại {METADATA_PATH}. Hãy chạy data_prep_colab.ipynb trước.'

os.chdir(PROJECT_DIR)
print(f'Project: {PROJECT_DIR}')
print(f'Dataset: {DATASET_DIR}')

## 4. Chia metadata thành train/valid manifest

In [ ]:
import json
import random

SEED = 1337
VALID_RATIO = 0.1
MANIFEST_DIR = DATASET_DIR / 'manifests'
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

with METADATA_PATH.open('r', encoding='utf-8') as f:
    records = json.load(f)

assert isinstance(records, list) and records, 'metadata.json phải là list và không được rỗng.'

rng = random.Random(SEED)
rng.shuffle(records)
valid_size = max(1, int(len(records) * VALID_RATIO))
valid_records = records[:valid_size]
train_records = records[valid_size:]

assert train_records, 'Không đủ dữ liệu train. Hãy tạo thêm sample hoặc giảm VALID_RATIO.'

train_manifest = MANIFEST_DIR / 'train.jsonl'
valid_manifest = MANIFEST_DIR / 'valid.jsonl'

def write_jsonl(path, items):
    with path.open('w', encoding='utf-8') as f:
        for item in items:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

write_jsonl(train_manifest, train_records)
write_jsonl(valid_manifest, valid_records)

print(f'Train samples: {len(train_records)} -> {train_manifest}')
print(f'Valid samples: {len(valid_records)} -> {valid_manifest}')
print('Example record:')
print(json.dumps(train_records[0], ensure_ascii=False, indent=2))

## 5. Cấu hình train

In [ ]:
# phase1_reconstruction: nên chạy trước.
# phase2_speaker_consistency: resume từ phase1/best.pt nếu muốn thêm speaker loss.
# phase3_negative_case: resume từ phase2/best.pt nếu muốn học absent-speaker case.
PHASE = 'phase1_reconstruction'

# Bắt đầu nhỏ để kiểm tra pipeline. Tăng EPOCHS/BATCH_SIZE sau khi chạy ổn.
EPOCHS = 2
BATCH_SIZE = 2
NUM_WORKERS = 2

# True: dùng enrollment wav và ECAPA online, dễ chạy ngay nhưng chậm hơn.
# False: cần manifest có embedding_path .npy.
USE_ONLINE_ECAPA = True

RUN_ROOT = Path('/content/drive/MyDrive/TSE_Dataset/runs')
OUTPUT_DIR = RUN_ROOT / PHASE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# True: mỗi lần chạy lại sẽ tự resume từ OUTPUT_DIR / 'last.pt' nếu file tồn tại.
AUTO_RESUME = True

# Điền đường dẫn checkpoint nếu muốn resume thủ công; giá trị này ưu tiên hơn AUTO_RESUME.
RESUME_FROM = None

print(f'Phase: {PHASE}')
print(f'Output: {OUTPUT_DIR}')

## 6. Chạy train

In [ ]:
import subprocess
import importlib

device = 'cuda' if torch.cuda.is_available() else 'cpu'
RUN_INLINE = True

script_args = [
    '--phase', PHASE,
    '--train-manifest', str(train_manifest),
    '--valid-manifest', str(valid_manifest),
    '--data-root', str(DATASET_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--device', device,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--num-workers', str(NUM_WORKERS),
]

if USE_ONLINE_ECAPA:
    script_args.append('--online-ecapa')

if AUTO_RESUME:
    script_args.append('--auto-resume')

if RESUME_FROM:
    script_args.extend(['--resume', str(RESUME_FROM)])

cmd = [sys.executable, '-u', 'train_personalized.py', *script_args]
should_train = True

auto_resume_ckpt = OUTPUT_DIR / 'last.pt'
if AUTO_RESUME and RESUME_FROM is None and auto_resume_ckpt.exists():
    ckpt = torch.load(str(auto_resume_ckpt), map_location='cpu')
    ckpt_epoch = int(ckpt.get('epoch', 0))
    print(f'Auto-resume checkpoint: {auto_resume_ckpt}')
    print(f'Checkpoint epoch={ckpt_epoch}, target EPOCHS={EPOCHS}')
    if ckpt_epoch >= EPOCHS:
        print('Nothing to train: checkpoint epoch is already >= EPOCHS. Increase EPOCHS or change OUTPUT_DIR for a fresh run.')
        should_train = False

print(' '.join(cmd))
if not should_train:
    print('Skipped training because checkpoint already satisfies target EPOCHS.')
elif RUN_INLINE:
    old_argv = sys.argv[:]
    sys.argv = ['train_personalized.py', *script_args]
    try:
        import train_personalized
        importlib.reload(train_personalized)
        train_personalized.main()
    finally:
        sys.argv = old_argv
    print('Train process finished inline')
else:
    result = subprocess.run(cmd, check=True)
    print(f'Train process finished with returncode={result.returncode}')

## 7. Kiểm tra checkpoint

In [ ]:
for name in ['best.pt', 'last.pt', 'config.json']:
    path = OUTPUT_DIR / name
    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f'{name}: {path} ({size_mb:.2f} MB)')
    else:
        print(f'{name}: chưa có')

## 8. Train phase 2


In [ ]:
# Chạy sau khi phase1_reconstruction đã có best.pt hoặc last.pt.
# Đây là số epoch train THÊM từ checkpoint phase 1.
PHASE2_MORE_EPOCHS = 40
PHASE2_BATCH_SIZE = BATCH_SIZE

RUN_INLINE = True

def best_or_last(output_dir):
    output_dir = Path(output_dir)
    for name in ['best.pt', 'last.pt']:
        path = output_dir / name
        if path.exists():
            return path
    raise FileNotFoundError(f'Không thấy best.pt/last.pt trong {output_dir}')

def checkpoint_epoch(path):
    ckpt = torch.load(str(path), map_location='cpu')
    return int(ckpt.get('epoch', 0))

def make_phase_start_checkpoint(resume_from, output_dir):
    resume_from = Path(resume_from)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    source_ckpt = torch.load(str(resume_from), map_location='cpu')
    source_epoch = int(source_ckpt.get('epoch', 0))
    phase_start = output_dir / 'phase_start.pt'
    torch.save({
        'model': source_ckpt['model'],
        'epoch': source_epoch,
        'best_metric': None,
        'bad_epochs': 0,
        'source_checkpoint': str(resume_from),
    }, str(phase_start))
    return phase_start, source_epoch

def run_phase2():
    phase = 'phase2_speaker_consistency'
    output_dir = RUN_ROOT / phase
    phase1_ckpt = best_or_last(RUN_ROOT / 'phase1_reconstruction')
    phase_start, source_epoch = make_phase_start_checkpoint(phase1_ckpt, output_dir)
    target_epochs = source_epoch + int(PHASE2_MORE_EPOCHS)
    resume_arg = phase_start

    own_last = output_dir / 'last.pt'
    if own_last.exists():
        own_epoch = checkpoint_epoch(own_last)
        print(f'{phase}: tìm thấy checkpoint riêng {own_last} epoch={own_epoch}')
        if own_epoch >= target_epochs:
            print(f'{phase}: đã đủ target_epochs={target_epochs}, bỏ qua train.')
            return best_or_last(output_dir)
        resume_arg = own_last

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    script_args = [
        '--phase', phase,
        '--train-manifest', str(train_manifest),
        '--valid-manifest', str(valid_manifest),
        '--data-root', str(DATASET_DIR),
        '--output-dir', str(output_dir),
        '--device', device,
        '--epochs', str(target_epochs),
        '--batch-size', str(PHASE2_BATCH_SIZE),
        '--num-workers', str(NUM_WORKERS),
        '--resume', str(resume_arg),
    ]

    if USE_ONLINE_ECAPA:
        script_args.append('--online-ecapa')

    cmd = [sys.executable, '-u', 'train_personalized.py', *script_args]
    print('\n' + '=' * 80)
    print(f'Phase: {phase}')
    print(f'Resume: {resume_arg} (source_epoch={source_epoch})')
    print(f'Target epochs: {target_epochs} = {source_epoch} + {PHASE2_MORE_EPOCHS}')
    print(f'Output: {output_dir}')
    print(' '.join(cmd))

    if RUN_INLINE:
        old_argv = sys.argv[:]
        sys.argv = ['train_personalized.py', *script_args]
        try:
            import train_personalized
            importlib.reload(train_personalized)
            train_personalized.main()
        finally:
            sys.argv = old_argv
    else:
        subprocess.run(cmd, check=True)

    return best_or_last(output_dir)

phase2_ckpt = run_phase2()
print(f'Phase2 checkpoint: {phase2_ckpt}')


## 9. Train phase 3


In [ ]:
# Chạy sau khi phase2_speaker_consistency đã có best.pt hoặc last.pt.
# Đây là số epoch train THÊM từ checkpoint phase 2.
PHASE3_MORE_EPOCHS = 20
PHASE3_BATCH_SIZE = BATCH_SIZE

RUN_INLINE = True

def best_or_last(output_dir):
    output_dir = Path(output_dir)
    for name in ['best.pt', 'last.pt']:
        path = output_dir / name
        if path.exists():
            return path
    raise FileNotFoundError(f'Không thấy best.pt/last.pt trong {output_dir}')

def checkpoint_epoch(path):
    ckpt = torch.load(str(path), map_location='cpu')
    return int(ckpt.get('epoch', 0))

def make_phase_start_checkpoint(resume_from, output_dir):
    resume_from = Path(resume_from)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    source_ckpt = torch.load(str(resume_from), map_location='cpu')
    source_epoch = int(source_ckpt.get('epoch', 0))
    phase_start = output_dir / 'phase_start.pt'
    torch.save({
        'model': source_ckpt['model'],
        'epoch': source_epoch,
        'best_metric': None,
        'bad_epochs': 0,
        'source_checkpoint': str(resume_from),
    }, str(phase_start))
    return phase_start, source_epoch

def count_negative_records(manifest_path):
    count = 0
    with Path(manifest_path).open('r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            case = str(item.get('case', '')).lower()
            if bool(item.get('is_negative', False)) or bool(item.get('negative', False)) or case in {'negative', 'absent', 'absent_speaker', 'wrong_speaker'}:
                count += 1
    return count

def run_phase3():
    phase = 'phase3_negative_case'
    output_dir = RUN_ROOT / phase
    phase2_source = globals().get('phase2_ckpt')
    if phase2_source is None:
        phase2_source = best_or_last(RUN_ROOT / 'phase2_speaker_consistency')
    phase_start, source_epoch = make_phase_start_checkpoint(phase2_source, output_dir)
    target_epochs = source_epoch + int(PHASE3_MORE_EPOCHS)
    resume_arg = phase_start

    own_last = output_dir / 'last.pt'
    if own_last.exists():
        own_epoch = checkpoint_epoch(own_last)
        print(f'{phase}: tìm thấy checkpoint riêng {own_last} epoch={own_epoch}')
        if own_epoch >= target_epochs:
            print(f'{phase}: đã đủ target_epochs={target_epochs}, bỏ qua train.')
            return best_or_last(output_dir)
        resume_arg = own_last

    neg_count = count_negative_records(train_manifest)
    if neg_count == 0:
        print('Warning: train manifest không có negative/absent-speaker record; phase3 sẽ gần giống phase2.')
    else:
        print(f'Negative train records: {neg_count}')

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    script_args = [
        '--phase', phase,
        '--train-manifest', str(train_manifest),
        '--valid-manifest', str(valid_manifest),
        '--data-root', str(DATASET_DIR),
        '--output-dir', str(output_dir),
        '--device', device,
        '--epochs', str(target_epochs),
        '--batch-size', str(PHASE3_BATCH_SIZE),
        '--num-workers', str(NUM_WORKERS),
        '--resume', str(resume_arg),
    ]

    if USE_ONLINE_ECAPA:
        script_args.append('--online-ecapa')

    cmd = [sys.executable, '-u', 'train_personalized.py', *script_args]
    print('\n' + '=' * 80)
    print(f'Phase: {phase}')
    print(f'Resume: {resume_arg} (source_epoch={source_epoch})')
    print(f'Target epochs: {target_epochs} = {source_epoch} + {PHASE3_MORE_EPOCHS}')
    print(f'Output: {output_dir}')
    print(' '.join(cmd))

    if RUN_INLINE:
        old_argv = sys.argv[:]
        sys.argv = ['train_personalized.py', *script_args]
        try:
            import train_personalized
            importlib.reload(train_personalized)
            train_personalized.main()
        finally:
            sys.argv = old_argv
    else:
        subprocess.run(cmd, check=True)

    return best_or_last(output_dir)

phase3_ckpt = run_phase3()
print(f'Phase3 checkpoint: {phase3_ckpt}')
